# FaPD, floor area

In [3]:
import pandas as pd
import numpy as np
import xarray as xr
from pathlib import Path

# Load Stock_TCJ from netCDF to preserve dims/coords exactly
stock_path = Path("Stock_TCJ.nc")
if not stock_path.exists():
    raise FileNotFoundError(
        "Stock_TCJ.nc not found. Run the save cell in Stock_TCJ.ipynb first."
    )
Stock_TCJ = xr.open_dataarray(stock_path).load()

# Clean/restore numeric cohort labels (drop non-numeric labels like 'year')
cohort_raw = Stock_TCJ.coords["cohort"].values
cohort_numeric = pd.to_numeric(cohort_raw, errors="coerce")
valid_cohort = ~np.isnan(cohort_numeric)
if not valid_cohort.all():
    Stock_TCJ = Stock_TCJ.isel(cohort=np.where(valid_cohort)[0])
    cohort_numeric = cohort_numeric[valid_cohort]
Stock_TCJ = Stock_TCJ.assign_coords(cohort=cohort_numeric.astype(int))

# Optional: make time numeric if possible
time_raw = Stock_TCJ.coords["time"].values
time_numeric = pd.to_numeric(time_raw, errors="coerce")
if not np.isnan(time_numeric).any():
    Stock_TCJ = Stock_TCJ.assign_coords(time=time_numeric.astype(int))

# Floor Area = Stock_TCJ * FApD[cohort, type]

# --- Last inn og bygg FApD for alle år 1600-2100 ---
df_fapd = pd.read_excel('BYGV06.xlsx', skiprows=2)

# Hent rådata 1916-2025 per type (rows 0-3, columns 1-110)
fapd_raw = {
    'Stuehus':   df_fapd.iloc[0, 1:111].values.astype(float),
    'Parcelhus': df_fapd.iloc[1, 1:111].values.astype(float),
    'Rekkehus':  df_fapd.iloc[2, 1:111].values.astype(float),
    'Etagehus':  df_fapd.iloc[3, 1:111].values.astype(float),
}

# Bygg FApD for alle kohorter i Stock_TCJ
cohort_years = Stock_TCJ.coords['cohort'].values.astype(int)

fapd_dict = {}
for btype, vals in fapd_raw.items():
    extended = []
    for year in cohort_years:
        if year < 1916:
            extended.append(vals[0])        # hold 1916-verdi konstant bakover
        elif year > 2025:
            extended.append(vals[-1])       # hold 2025-verdi konstant fremover
        else:
            extended.append(vals[year - 1916])
    fapd_dict[btype] = extended

# Bygg FApD som xarray DataArray [cohort, type]
FApD = xr.DataArray(
    np.array([fapd_dict[t] for t in Stock_TCJ.coords['type'].values]).T,
    dims=['cohort', 'type'],
    coords={
        'cohort': Stock_TCJ.coords['cohort'],
        'type':   Stock_TCJ.coords['type'],
    }
)

# --- Beregn floor area ---
# Stock_TCJ [dwellings] * FApD [m²/dwelling] = Floor_area_TCJ [m²]
Floor_area_TCJ = Stock_TCJ * FApD

# --- Sanity check ---
print("Loaded Stock_TCJ from:", stock_path.resolve())
print("Stock_TCJ dims:", Stock_TCJ.dims)
print("FApD sample (parcelhus 1950-2000):")
print(FApD.sel(type='Parcelhus', cohort=slice(1950, 2000)).values[::10])

print("\nFloor area total per type in 2020 [m²]:")
if 2020 in Floor_area_TCJ.coords['time'].values:
    print(Floor_area_TCJ.sel(time=2020).sum(dim='cohort').to_pandas().round(0))
else:
    print("Time coordinate does not include year labels (e.g. 2020).")

Loaded Stock_TCJ from: C:\Users\Bruker\OneDrive\TEP4290\Project\TEP4290-project\Stock_TCJ.nc
Stock_TCJ dims: ('time', 'cohort', 'type')
FApD sample (parcelhus 1950-2000):
[120. 115. 149. 149. 137. 173.]

Floor area total per type in 2020 [m²]:
Time coordinate does not include year labels (e.g. 2020).


# Material Intensity

In [ ]:


materials = ['concrete', 'brick', 'wood', 'steel', 'glass', 'aluminum', 'copper']

# Definer hvilke (function, structure) som mapper til hvilken bygningstype
type_mapping = {
    'stuehus':   [('RS', 'M')],
    'parcelhus': [('RS', 'M'), ('RS', 'T')],
    'rekkehus':  [('RS', 'M')],
    'etagehus':  [('RM', 'C'), ('RM', 'M')],
}

mi_matrix = {}  # {bygningstype: {materiale: median_verdi}}

for mat in materials:
    df = pd.read_excel('MI_data_20230905.xlsx', sheet_name=mat)
    df_eu15 = df[df['R5_32'] == 'OECD_EU15']
    
    for building_type, combos in type_mapping.items():
        mask = df_eu15.apply(
            lambda row: (row['function'], row['structure']) in combos, axis=1
        )
        subset = df_eu15[mask][mat].dropna()
        
        median_val = subset.median()
        
        if building_type not in mi_matrix:
            mi_matrix[building_type] = {}
        mi_matrix[building_type][mat] = round(median_val, 1)

# Skriv ut som tabell
mi_df = pd.DataFrame(mi_matrix).T
print(mi_df)
mi_df.to_excel('MI_matrix_RASMI.xlsx')

           concrete  brick  wood  steel  glass  aluminum  copper
stuehus      1014.5  693.3  20.3   70.5    4.1       2.0     NaN
parcelhus     479.0  635.3  54.0   48.0    2.0       2.0     9.0
rekkehus     1014.5  693.3  20.3   70.5    4.1       2.0     NaN
etagehus      773.0  390.6  21.9   42.1    3.0       2.9     0.2
